In [1]:
###PYTHON VERSION

import os


# Force PyTorch to map sm_90 kernels to your Blackwell GPU
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"

# Keep your existing configs...
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Helps reduce PyTorch memory fragmentation.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import json
import re
import time
from pathlib import Path
from typing import Optional

import torch
import transformers
import vllm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm.auto import tqdm

print("Python executable:", sys.executable)
print("CUDA available:", torch.cuda.is_available())
print("CUDA visible device count:", torch.cuda.device_count())
print("CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)
print("transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"cuda:{i} ->", torch.cuda.get_device_name(i))
else:
    raise RuntimeError("CUDA is not available. You are not in a GPU pod/session.")

Python executable: /home/amn024/private/151B_SP26_Competition/.venv/bin/python
CUDA available: True
CUDA visible device count: 1
CUDA version: 12.1
Torch version: 2.5.1+cu121
transformers: 5.9.0
vLLM: 0.7.3
cuda:0 -> NVIDIA A30


In [2]:
print("=== STARTING HARDWARE FUNCTIONALITY CHECK ===")

try:
    # 1. Initialize random 1000x1000 floating-point matrices directly inside the GPU VRAM.
    # This verifies that PyTorch can successfully allocate tensor memory on the Blackwell architecture.
    print("Allocating test matrices on GPU (cuda:0)...")
    matrix_a = torch.randn(1000, 1000, device="cuda")
    matrix_b = torch.randn(1000, 1000, device="cuda")

    # 2. Perform a heavy matrix multiplication (GEMM operations).
    # This forces the NVIDIA hardware driver to compile and execute raw CUDA kernels,
    # proving that the sm_120 hardware is successfully processing code targeting sm_90.
    print("Executing matrix multiplication CUDA kernels...")
    result_matrix = torch.matmul(matrix_a, matrix_b)

    # 3. Synchronize the CUDA device to ensure operations finish without silent background failures.
    torch.cuda.synchronize()
    
    print("\n[SUCCESS] Pipeline is 100% operational!")
    print(f"-> Verified: CUDA is executing operations successfully.")
    print(f"-> Output Tensor Shape: {result_matrix.shape}")
    print("-> Status: You can safely ignore the architecture warning. Your GPU is active and ready.")

except Exception as error:
    print("\n[FAILURE] Hardware check failed. See error details below:")
    print(str(error))

print("=============================================")

=== STARTING HARDWARE FUNCTIONALITY CHECK ===
Allocating test matrices on GPU (cuda:0)...
Executing matrix multiplication CUDA kernels...

[SUCCESS] Pipeline is 100% operational!
-> Verified: CUDA is executing operations successfully.
-> Output Tensor Shape: torch.Size([1000, 1000])
-> Status: You can safely ignore the architecture warning. Your GPU is active and ready.


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

DATA_PATH = "data/public.jsonl"

RUN_NAME = "k=5test"
# # K=1 baseline (sampling, no voting)
# RUN_NAME = "prompt_v2_sc_k1_50";  n=1

# # K=3 self-consistency
# RUN_NAME = "prompt_v2_sc_k3_50";  n=3

# # K=5 self-consistency
# RUN_NAME = "prompt_v2_sc_k5_50";  n=5
OUTPUT_PATH = f"results/{RUN_NAME}.jsonl"

# Conservative first. After it works, raise this to 8192.
MAX_TOKENS = 16384 
# qwen say suse 81k too much for A30 try 32k

# Start with 10. After model loads + scores correctly, change to 50.
EVAL_LIMIT = 200

print("MODEL_ID:", MODEL_ID)
print("DATA_PATH:", DATA_PATH)
print("RUN_NAME:", RUN_NAME)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("MAX_TOKENS:", MAX_TOKENS)
print("EVAL_LIMIT:", EVAL_LIMIT)

MODEL_ID: Qwen/Qwen3-4B-Thinking-2507
DATA_PATH: data/public.jsonl
RUN_NAME: k=5test
OUTPUT_PATH: results/k=5test.jsonl
MAX_TOKENS: 16384
EVAL_LIMIT: 200


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data_path = Path(DATA_PATH)
assert data_path.exists(), f"Cannot find {DATA_PATH}. Run this notebook from the competition repo root."

data = [json.loads(line) for line in open(data_path, encoding="utf-8")]

if EVAL_LIMIT is None:
    eval_data = data
else:
    eval_data = data[:EVAL_LIMIT]

n_mcq_all  = sum(bool(d.get("options")) for d in data)
n_free_all = sum(not d.get("options") for d in data)

n_mcq_eval  = sum(bool(d.get("options")) for d in eval_data)
n_free_eval = sum(not d.get("options") for d in eval_data)

print(f"Loaded {len(data)} total questions  ({n_mcq_all} MCQ, {n_free_all} free-form)")
print(f"Evaluating {len(eval_data)} questions ({n_mcq_eval} MCQ, {n_free_eval} free-form)")

# Preview one MCQ and one free-form item from eval_data
mcq_sample  = next(d for d in eval_data if d.get("options"))
free_sample = next(d for d in eval_data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1500])
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1500])


Loaded 1126 total questions  (375 MCQ, 751 free-form)
Evaluating 200 questions (68 MCQ, 132 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
# Goal: force a final boxed answer while avoiding endless reasoning loops.

# Official Qwen CoT trigger phrase — matches Qwen2.5-Math training distribution exactly
SYSTEM_PROMPT_FREEFORM = """Please reason step by step, and put your final answer within \\boxed{}.
If the problem asks for multiple values or has multiple fill-in-the-blank placeholders, list all answers in order inside a single \\boxed{}, separated by commas, e.g. \\boxed{3, 7}.""".strip()

SYSTEM_PROMPT_MCQ = """Please reason step by step, and put your final answer within \\boxed{}.
Your boxed answer must contain exactly one capital letter representing the correct choice, e.g. \\boxed{C}.""".strip()

#SYSTEM_PROMPT_FREEFORM = """
#Please reason step by step, and put your final answer within \\boxed{}.

#Formatting rules:
#1. After thorough verification, your absolute final line must be exactly: Therefore, the answer is \\boxed{...}
#2. Put only the clean, final answer content inside \\boxed{}.
#3. If the problem has multiple [ANS] blanks, put the answers in order, separated by commas inside the single box.
#4. Do not use words like "approximately" unless the problem explicitly asks for an approximation.
#""".strip()

#SYSTEM_PROMPT_MCQ = """
#Please reason step by step, and put your final answer within \\boxed{}.

#Formatting rules:
#1. The absolute final line must be exactly: Therefore, the answer is \\boxed{X}
#2. X must be exactly one capital letter representing the choice (e.g., A, B, C, D, or E).
#3. Do not put the full option text or anything else inside \\boxed{}.
#""".strip()


#SYSTEM_PROMPT_FREEFORM = """
#You are an expert mathematician. Solve this problem using a highly detailed, step-by-step Chain of Thought. 

#Break the problem down into logical sub-tasks. At the end of each major step, rigorously validate your reasoning and arithmetic to ensure no calculation or conceptual errors have occurred. If you detect an inconsistency, backtrack and correct it immediately. 

#Formatting rules:
#1. After thorough verification, your absolute final line must be exactly: Therefore, the answer is \\boxed{...}
#2. Put only the clean, final answer content inside \\boxed{}.
#3. If the problem has multiple [ANS] blanks, put the answers in order, separated by commas inside the single box.
#4. Do not use words like "approximately" unless the problem explicitly asks for an approximation.
#""".strip()

#SYSTEM_PROMPT_MCQ = """
#You are an expert mathematician. Use a rigorous Chain of Thought approach to solve this multiple-choice problem.

#First, read and analyze the problem independently without looking at the choices. Derive your result step by step, and validate each stage of your deduction and arithmetic. Once your independent derivation is fully verified, compare your final result against the provided options. If your result does not match any option, re-examine your assumptions and backtrack immediately.

#Formatting rules:
#1. The absolute final line must be exactly: Therefore, the answer is \\boxed{X}
#2. X must be exactly one capital letter representing the choice (e.g., A, B, C, D, or E).
#3. Do not put the full option text or anything else inside \\boxed{}.
#""".strip()

#SYSTEM_PROMPT_FREEFORM = """
#You are a careful math solver.
#
#Solve the problem step by step, but keep the reasoning concise.
#Do not stop before giving the final answer.

#Formatting rules:
#1. The final line must be exactly: Therefore, the answer is \\boxed{...}
#2. Put only the final answer content inside \\boxed{}.
#3. If the problem has multiple [ANS] blanks, put the answers in order, separated by commas.
#4. Do not use words like "approximately" unless the problem asks for an approximation.
#""".strip()

#SYSTEM_PROMPT_MCQ = """
#You are a careful math solver.

#Solve the multiple-choice problem step by step, but keep the reasoning concise.
#Compare your result to the answer choices.

#Formatting rules:
#1. The final line must be exactly: Therefore, the answer is \\boxed{X}
#2. X must be one capital letter such as A, B, C, D, or E.
#3. Do not put the full option text inside \\boxed{}.
#""".strip()


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for one competition item."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{label}. {str(option).strip()}"
            for label, option in zip(labels, options)
        )

        user_prompt = f"""
Problem:
{question}

Answer choices:
{opts_text}

Solve the problem and end with the required boxed letter.
""".strip()

        return SYSTEM_PROMPT_MCQ, user_prompt

    user_prompt = f"""
Problem:
{question}

Solve the problem and end with the required boxed answer.
""".strip()

    return SYSTEM_PROMPT_FREEFORM, user_prompt

In [6]:
# ── Python (Program of Thought) System Prompts ───────────────────────────────
# Lead phrase mirrors official Qwen2.5-Math TIR training distribution for best model alignment
SYSTEM_PROMPT_PYTHON_FREEFORM = """Please integrate natural language reasoning with programs to solve the problem above, and put your final answer within \\boxed{}.

Write a self-contained Python script to compute the answer:
1. Use sympy for symbolic/exact answers; numpy or math for numerical computation
2. Your script's LAST print() must output ONLY the answer value — no labels, no units, no extra text
3. If the problem has multiple fill-in-the-blank placeholders, print all answers comma-separated on one line
4. Wrap your code in ```python ... ``` blocks""".strip()

SYSTEM_PROMPT_PYTHON_MCQ = """Please integrate natural language reasoning with programs to solve the problem above, and put your final answer within \\boxed{}.

Write a self-contained Python script to derive the answer and identify the matching choice:
1. Use sympy for exact symbolic computation
2. Your script's LAST print() must output ONLY the single capital letter of the correct choice (e.g. C)
3. Wrap your code in ```python ... ``` blocks""".strip()


def build_python_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{label}. {str(opt).strip()}" for label, opt in zip(labels, options))
        user_prompt = (
            f"Problem:\n{question}\n\nAnswer choices:\n{opts_text}\n\n"
            "Write Python code to solve this. The last print() must output only the correct letter."
        )
        return SYSTEM_PROMPT_PYTHON_MCQ, user_prompt

    user_prompt = (
        f"Problem:\n{question}\n\n"
        "Write Python code to solve this. The last print() must output only the final answer."
    )
    return SYSTEM_PROMPT_PYTHON_FREEFORM, user_prompt


def build_python_retry_prompt(
    question: str, options: Optional[list], prev_code: str, error: str
) -> tuple[str, str]:
    sys_p, _ = build_python_prompt(question, options)
    user_prompt = (
        f"Problem:\n{question}\n\n"
        f"Your previous Python attempt failed with this error:\n{error}\n\n"
        f"Failed code:\n```python\n{prev_code}\n```\n\n"
        "Fix the error and write a correct Python solution. "
        "The last print() must output only the final answer "
        "(for multiple answers, comma-separated on one line)."
    )
    return sys_p, user_prompt


print("Python PoT prompts loaded.")

Python PoT prompts loaded.


## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [7]:
# ── Load tokenizer + patch Qwen tokenizer compatibility ──────────────────────
from transformers.models.qwen2.tokenization_qwen2 import Qwen2Tokenizer

if not hasattr(Qwen2Tokenizer, "all_special_tokens_extended"):
    print("Patching Qwen2Tokenizer.all_special_tokens_extended ...")

    @property
    def all_special_tokens_extended(self):
        return list(self.all_special_tokens)

    Qwen2Tokenizer.all_special_tokens_extended = all_special_tokens_extended
else:
    print("Qwen2Tokenizer already has all_special_tokens_extended.")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="left",
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer class:", tokenizer.__class__)
print("Has all_special_tokens_extended:", hasattr(tokenizer, "all_special_tokens_extended"))

# ── Load vLLM model ───────────────────────────────────────────────────────────
vllm_model = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    trust_remote_code=True,
    gpu_memory_utilization=0.92,
    max_model_len=32768,
    max_num_seqs=16,
    max_num_batched_tokens=32768,
    enable_chunked_prefill=True,
    enable_prefix_caching=True,
)

# Reasoning fallback sampling — official Qwen3-Thinking recommended params
# presence_penalty removed: not in Qwen3 official param set, can truncate <think> blocks early
sampling_params_sc = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    n=1,
    repetition_penalty=1.0,
)

print("Model loaded.")

Patching Qwen2Tokenizer.all_special_tokens_extended ...


Tokenizer class: <class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
Has all_special_tokens_extended: True
INFO 05-25 16:06:09 __init__.py:207] Automatically detected platform cuda.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


INFO 05-25 16:06:18 config.py:549] This model supports multiple tasks: {'score', 'generate', 'embed', 'classify', 'reward'}. Defaulting to 'generate'.


INFO 05-25 16:06:18 config.py:1555] Chunked prefill is enabled with max_num_batched_tokens=32768.


...

Capturing CUDA graph shapes: 100%|██████████| 5/5 [00:02<00:00,  1.74it/s]

Capturing CUDA graph shapes: 100%|██████████| 5/5 [00:02<00:00,  1.73it/s]

INFO 05-25 16:06:35 model_runner.py:1562] Graph capturing finished in 3 secs, took 0.12 GiB


INFO 05-25 16:06:35 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 12.47 seconds


Model loaded.


In [8]:
# ── Python Execution Utilities ────────────────────────────────────────────────
import subprocess
import tempfile
import os


def extract_python_code(text: str) -> Optional[str]:
    """Extract code from the first ```python ... ``` block in model output."""
    m = re.search(r'```python\s*(.*?)\s*```', text, re.DOTALL)
    if m:
        return m.group(1).strip()
    # Fallback: any ``` block that contains a print() call
    m = re.search(r'```\s*(.*?)\s*```', text, re.DOTALL)
    if m:
        code = m.group(1).strip()
        if 'print' in code:
            return code
    return None


def execute_python(code: str, timeout: int = 15) -> tuple[Optional[str], Optional[str]]:
    """
    Run code in a subprocess. Returns (stdout, error_msg) — exactly one is None.
    Truncates stderr to the last 500 chars to keep error feedback concise.
    """
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False, encoding='utf-8') as f:
        f.write(code)
        tmp_path = f.name
    try:
        proc = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True, text=True, timeout=timeout,
        )
        os.unlink(tmp_path)
        if proc.returncode == 0:
            out = proc.stdout.strip()
            return (out, None) if out else (None, "Script produced no output.")
        return None, proc.stderr.strip()[-500:]
    except subprocess.TimeoutExpired:
        try: os.unlink(tmp_path)
        except: pass
        return None, f"Timed out after {timeout}s."
    except Exception as e:
        try: os.unlink(tmp_path)
        except: pass
        return None, str(e)


print("Python execution utilities loaded.")

Python execution utilities loaded.


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [9]:
# ── PoT Generation Pipeline ────────────────────────────────────────────────────
# Phase 1: Python code generation + execution (batched, up to MAX_PYTHON_RETRIES+1 rounds)
# Phase 2: Reasoning fallback for questions where Python failed (batched, K samples)
# Output:  per_question_raw — same dict format as before, fully compatible with scoring cell.

PYTHON_TIMEOUT     = 15  # seconds per subprocess execution
MAX_PYTHON_RETRIES = 4   # retry attempts after first failure (feeds error back to model)


def format_chat_prompt(item: dict) -> str:
    """Reasoning prompt used by the fallback path."""
    system, user = build_prompt(item["question"], item.get("options"))
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )


def format_python_prompt(item: dict, prev_code: str = None, error: str = None) -> str:
    """Python code generation prompt; includes error context on retry."""
    if prev_code is not None and error is not None:
        system, user = build_python_retry_prompt(
            item["question"], item.get("options"), prev_code, error
        )
    else:
        system, user = build_python_prompt(item["question"], item.get("options"))
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )


# temperature=0.6: Qwen3-Thinking official minimum — below this degrades thinking quality
# max_tokens=8192: thinking block alone can consume 1-2k tokens on hard problems; 4096 was too tight
sampling_params_python = SamplingParams(
    max_tokens=8192,
    temperature=0.6,
    top_p=0.95,
    n=1,
    repetition_penalty=1.0,
)

# ── State ─────────────────────────────────────────────────────────────────────
pending   = list(range(len(eval_data)))  # question indices awaiting a Python success
py_state  = {}                           # idx -> {"code": str, "error": str} for retry prompts
final_raw = {}                           # idx -> finalized samples list

# ── Python rounds (batched) ───────────────────────────────────────────────────
for attempt in range(MAX_PYTHON_RETRIES + 1):
    if not pending:
        break

    print(f"\n── Python attempt {attempt + 1}/{MAX_PYTHON_RETRIES + 1}  ({len(pending)} questions) ──")

    py_prompts = [
        format_python_prompt(
            eval_data[idx],
            prev_code=py_state.get(idx, {}).get("code"),
            error=py_state.get(idx, {}).get("error"),
        )
        for idx in pending
    ]
    py_outputs = vllm_model.generate(py_prompts, sampling_params=sampling_params_python)

    still_pending = []
    for idx, out in zip(pending, py_outputs):
        resp  = out.outputs[0].text.strip()
        n_tok = len(out.outputs[0].token_ids)
        code  = extract_python_code(resp)

        if code is None:
            py_state[idx] = {"code": "", "error": "No ```python``` block found in response."}
            still_pending.append(idx)
            continue

        stdout, err = execute_python(code, timeout=PYTHON_TIMEOUT)

        if stdout is not None:
            # Wrap in \boxed{} so extract_boxed + scoring cell work with zero changes
            final_raw[idx] = [{
                "text": f"\\boxed{{{stdout.strip()}}}",
                "n_tokens": n_tok,
                "finish_reason": "stop",
                "source": "python",
            }]
        else:
            py_state[idx] = {"code": code, "error": err}
            still_pending.append(idx)

    pending = still_pending
    print(f"   Successes so far: {len(final_raw)}/{len(eval_data)}  |  pending: {len(pending)}")

# ── Reasoning fallback (batched, K samples) ───────────────────────────────────
if pending:
    print(f"\n── Reasoning fallback for {len(pending)} questions ──")
    fb_prompts = [format_chat_prompt(eval_data[idx]) for idx in pending]
    fb_outputs = vllm_model.generate(fb_prompts, sampling_params=sampling_params_sc)

    for idx, out in zip(pending, fb_outputs):
        final_raw[idx] = [
            {
                "text": o.text.strip(),
                "n_tokens": len(o.token_ids),
                "finish_reason": o.finish_reason,
                "source": "reasoning",
            }
            for o in out.outputs
        ]

# ── Reconstruct per_question_raw in original order ────────────────────────────
per_question_raw = [final_raw[i] for i in range(len(eval_data))]

assert len(per_question_raw) == len(eval_data)

n_python   = sum(1 for s in per_question_raw if s[0].get("source") == "python")
n_fallback = sum(1 for s in per_question_raw if s[0].get("source") == "reasoning")
K = max(len(s) for s in per_question_raw)

print(f"\nPipeline complete.")
print(f"  Python path : {n_python}/{len(eval_data)}")
print(f"  Reasoning   : {n_fallback}/{len(eval_data)}")
print(f"  K (max)     : {K}")
print(f"\nSample 0 source : {per_question_raw[0][0].get('source')}")
print(f"Sample 0 text   : {per_question_raw[0][0]['text'][:300]}")


── Python attempt 1/4  (200 questions) ──


Processed prompts:   0%|          | 0/200 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 1/200 [00:40<2:14:45, 40.63s/it, est. speed input: 10.73 toks/s, output: 40.69 toks/s]

Processed prompts:   1%|          | 2/200 [00:45<1:04:28, 19.54s/it, est. speed input: 13.19 toks/s, output: 76.66 toks/s]

Processed prompts:   2%|▏         | 3/200 [01:18<1:25:10, 25.94s/it, est. speed input: 10.00 toks/s, output: 82.23 toks/s]

...

Processed prompts:  91%|█████████ | 20/22 [17:44<01:39, 49.86s/it, est. speed input: 6.99 toks/s, output: 228.46 toks/s]

Processed prompts:  95%|█████████▌| 21/22 [17:45<00:35, 35.47s/it, est. speed input: 9.77 toks/s, output: 243.80 toks/s]

Processed prompts: 100%|██████████| 22/22 [20:26<00:00, 72.32s/it, est. speed input: 8.89 toks/s, output: 225.09 toks/s]

Processed prompts: 100%|██████████| 22/22 [20:26<00:00, 55.74s/it, est. speed input: 8.89 toks/s, output: 225.09 toks/s]


Pipeline complete.
  Python path : 178/200
  Reasoning   : 22/200
  K (max)     : 1

Sample 0 source : python
Sample 0 text   : \boxed{105950}


In [14]:
def extract_boxed(text: str):
    """
    Extract the last \\boxed{...} content.
    This handles nested braces like \\boxed{\\frac{1}{2}}, unlike a simple regex.
    """
    marker = r"\boxed{"
    start = text.rfind(marker)
    if start == -1:
        return None

    i = start + len(marker)
    depth = 1
    chars = []

    while i < len(text):
        ch = text[i]

        if ch == "{":
            depth += 1
            chars.append(ch)
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return "".join(chars).strip()
            chars.append(ch)
        else:
            chars.append(ch)

        i += 1

    return None


# for i in range(min(5, len(responses))):
#     print("=" * 80)
#     print("id:", eval_data[i].get("id"))
#     print("boxed:", extract_boxed(responses[i]))
#     print("response length:", len(responses[i]))
#     print("tail:")
#     print(responses[i][-1000:])


In [15]:
import re

def extract_letter(text: str) -> str:
    """
    Backup helper to find a capital letter (A-E) in the text 
    if the boxed extraction fails or is empty.
    """
    if not text:
        return ""
        
    # 1. Look for common patterns like "The answer is A" or "Choice: B"
    patterns = [
        r"answer is ([A-E])",
        r"answer is: ([A-E])",
        r"answer is \(([A-E])\)",
        r"Choice ([A-E])",
        r"Option ([A-E])",
    ]
    
    for pattern in patterns:
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()
            
    # 2. Last resort: check the very end of the text for any standalone A-E
    # (Often models end with "Therefore, the answer is B.")
    last_bit = text[-50:].upper()
    m = re.search(r"\b([A-E])\b", last_bit)
    if m:
        return m.group(1).upper()
        
    return ""

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [16]:
# ── Vote + score + record diagnostics ─────────────────
def majority_vote(boxed_answers):
    """Return (voted_answer, status). Status: 'majority' | 'tie_first' | 'all_none'."""
    valid = [b for b in boxed_answers if b is not None]
    if not valid:
        return None, "all_none"
    counts = {}
    for b in valid:
        counts[b] = counts.get(b, 0) + 1
    max_count = max(counts.values())
    winners = {b for b, c in counts.items() if c == max_count}
    if len(winners) == 1:
        return next(iter(winners)), "majority"
    # Tie: deterministic — first occurrence wins
    for b in valid:
        if b in winners:
            return b, "tie_first"

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, samples in tqdm(zip(eval_data, per_question_raw), total=len(eval_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold = item.get("answer", None)
    source = samples[0].get("source", "unknown")

    sample_texts = [s["text"] for s in samples]
    sample_boxed = [extract_boxed(t) for t in sample_texts]

    if len(samples) == 1:
        voted, vote_status = sample_boxed[0], "single"
    else:
        voted, vote_status = majority_vote(sample_boxed)

    # Pick representative trace (the one whose boxed matches the vote)
    if voted is not None:
        rep_idx = next((i for i, b in enumerate(sample_boxed) if b == voted), 0)
    else:
        rep_idx = 0
    rep_text = sample_texts[rep_idx]

    # Score against the VOTED answer (not just sample 0)
    if gold is None:
        correct = None
    elif is_mcq:
        if voted is not None:
            m = re.search(r"\b([A-Z])\b", voted.strip().upper())
            pred_letter = m.group(1) if m else extract_letter(rep_text)
        else:
            pred_letter = extract_letter(rep_text)
        correct = (pred_letter == str(gold).strip().upper())
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=rep_text,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    # Diagnostics across the K samples
    has_boxed_per  = [b is not None for b in sample_boxed]
    truncated_per  = [s["finish_reason"] == "length" for s in samples]
    n_tokens_per   = [s["n_tokens"] for s in samples]

    results.append({
        "id": item.get("id"),
        "is_mcq": is_mcq,
        "gold": gold,
        "K": len(samples),
        "source": source,
        "samples_boxed": sample_boxed,
        "voted": voted,
        "vote_status": vote_status,
        "rep_response": rep_text,
        "correct": correct,
        "any_has_boxed":  any(has_boxed_per),
        "all_have_boxed": all(has_boxed_per),
        "any_truncated":  any(truncated_per),
        "all_truncated":  all(truncated_per),
        "tokens_per_sample": n_tokens_per,
        "max_tokens_used": max(n_tokens_per),
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 0/200 [00:00<?, ?it/s]

Scoring complete. 200 results.


## 8. Summary

Print accuracy broken down by question type.

In [17]:
scored_results = [r for r in results if r["correct"] is not None]
mcq_res  = [r for r in scored_results if r["is_mcq"]]
free_res = [r for r in scored_results if not r["is_mcq"]]

def acc(subset):
    return sum(bool(r["correct"]) for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 60)
print("EVALUATION RESULTS")
print("RUN_NAME:", RUN_NAME)
print("=" * 60)
print(f"  MCQ        : {sum(bool(r['correct']) for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(bool(r['correct']) for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(bool(r['correct']) for r in scored_results):4d} / {len(scored_results):4d}  ({acc(scored_results):.2f}%)")
print("=" * 60)


EVALUATION RESULTS
RUN_NAME: k=5test
  MCQ        :   42 /   68  (61.76%)
  Free-form  :   77 /  132  (58.33%)
  Overall    :  119 /  200  (59.50%)


In [18]:
# ── Formatting diagnostics ────────────────────────────────────────────────────
def format_diagnostics(results):
    n = len(results)
    has_any   = sum(1 for r in results if r["any_has_boxed"])
    has_all   = sum(1 for r in results if r["all_have_boxed"])
    miss_all  = sum(1 for r in results if not r["any_has_boxed"])
    trunc_any = sum(1 for r in results if r["any_truncated"])
    trunc_all = sum(1 for r in results if r["all_truncated"])
    ties      = sum(1 for r in results if r.get("vote_status") == "tie_first")
    none_vote = sum(1 for r in results if r.get("vote_status") == "all_none")
    avg_tok   = sum(sum(r["tokens_per_sample"]) / len(r["tokens_per_sample"]) for r in results) / n
    n_python  = sum(1 for r in results if r.get("source") == "python")
    n_reason  = sum(1 for r in results if r.get("source") == "reasoning")

    pct = lambda x: f"{x}/{n} ({x/n*100:.1f}%)"
    return {
        "RUN_NAME": RUN_NAME,
        "n": n,
        "Python path":             pct(n_python),
        "Reasoning fallback":      pct(n_reason),
        "Has Boxed (any sample)":  pct(has_any),
        "Has Boxed (all samples)": pct(has_all),
        "Missing Boxed (all)":     pct(miss_all),
        "Truncated (any sample)":  pct(trunc_any),
        "Truncated (all samples)": pct(trunc_all),
        "Vote ties":               pct(ties),
        "All-None votes":          pct(none_vote),
        "Avg tokens/sample":       round(avg_tok, 1),
    }

diag = format_diagnostics(results)
print("=" * 70)
print("FORMATTING DIAGNOSTICS")
print("=" * 70)
for k, v in diag.items():
    print(f"  {k:30s} : {v}")
print("=" * 70)

FORMATTING DIAGNOSTICS
  RUN_NAME                       : k=5test
  n                              : 200
  Python path                    : 178/200 (89.0%)
  Reasoning fallback             : 22/200 (11.0%)
  Has Boxed (any sample)         : 193/200 (96.5%)
  Has Boxed (all samples)        : 193/200 (96.5%)
  Missing Boxed (all)            : 7/200 (3.5%)
  Truncated (any sample)         : 8/200 (4.0%)
  Truncated (all samples)        : 8/200 (4.0%)
  Vote ties                      : 0/200 (0.0%)
  All-None votes                 : 0/200 (0.0%)
  Avg tokens/sample              : 6016.0


In [19]:
print("len(data):", len(data))
print("len(eval_data):", len(eval_data))
print("len(prompts):", len(prompts))
print("len(responses):", len(responses))
print("len(results):", len(results))

# Show wrong examples to diagnose prompt failures.
wrong = [r for r in results if r["correct"] is False]

print("wrong count:", len(wrong))

for r in wrong[:5]:
    print("=" * 100)
    print("id:", r["id"], "is_mcq:", r["is_mcq"], "gold:", r["gold"], "boxed:", r["boxed"])
    print("response tail:")
    print(r["response"][-1200:])


len(data): 1126
len(eval_data): 200


NameError: name 'prompts' is not defined

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!